<a href="https://colab.research.google.com/github/play71214-afk/Atlas-de-particulas/blob/main/ProyectoEspacial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import io, os, json, base64, urllib.request
import numpy as np
import plotly.graph_objects as go
from PIL import Image, ImageDraw
from google.colab import files

rng = np.random.default_rng(42)

def estilo_oscuro(fig, titulo):
    ejes = dict(visible=False)
    fig.update_layout(
        title=dict(text=titulo, font=dict(color="white", size=20)),
        paper_bgcolor="black", plot_bgcolor="black", uirevision="atlas",
        scene=dict(xaxis=ejes, yaxis=ejes, zaxis=ejes, bgcolor="black", aspectmode="data"),
        legend=dict(font=dict(color="white"), bgcolor="rgba(0,0,0,0)"),
        margin=dict(l=0, r=0, t=50, b=0), height=None)
    return fig

# ---------------- VÍA LÁCTEA ----------------
def via_lactea(n_disco=25000, n_bulbo=5000, n_brazos=4, radio=15):
    r = rng.exponential(radio / 3.5, n_disco); r = r[r < radio]
    th = 2*np.pi*rng.integers(0, n_brazos, len(r))/n_brazos + 0.4*r + rng.normal(0, 0.25, len(r))
    x = r*np.cos(th) + rng.normal(0, 0.4, len(r))
    y = r*np.sin(th) + rng.normal(0, 0.4, len(r))
    z = rng.normal(0, 0.05 + 0.3*(1 - r/radio), len(r))
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode="markers", name="Estrellas del disco",
        marker=dict(size=1.5, color=r, opacity=0.8,
                    colorscale=[[0, "#fff3c4"], [0.5, "#9ecbff"], [1, "#3d5aff"]]), hoverinfo="skip"))
    fig.add_trace(go.Scatter3d(x=rng.normal(0, 1.6, n_bulbo), y=rng.normal(0, 1.6, n_bulbo),
        z=rng.normal(0, 1.0, n_bulbo), mode="markers", name="Bulbo galáctico",
        marker=dict(size=1.8, color="#ffd27a", opacity=0.7), hoverinfo="skip"))
    fig.add_trace(go.Scatter3d(x=[8.2], y=[0], z=[0], mode="markers+text", name="Sol",
        text=["☀ Clic aquí: Sistema Solar"], textposition="top center", textfont=dict(color="white"),
        marker=dict(size=16, color="#ffb300", line=dict(color="white", width=2)),
        hovertemplate="<b>Sol</b><br>Haz clic para entrar al Sistema Solar<extra></extra>"))
    return estilo_oscuro(fig, "Vía Láctea")

# ---------------- SISTEMA SOLAR ----------------
#            UA,  incl°, tamaño, color,    ángulo°
planetas = {
    "Mercurio": (0.39, 7.0, 7, "#b1b1b1", 20),  "Venus":   (0.72, 3.4, 9, "#e8c07d", 110),
    "Tierra":   (1.00, 0.0, 9, "#3b82f6", 200), "Marte":   (1.52, 1.85, 8, "#e2573b", 290),
    "Júpiter":  (5.20, 1.3, 18, "#d9a066", 60), "Saturno": (9.54, 2.5, 16, "#e6d29a", 150),
    "Urano":    (19.2, 0.8, 13, "#8fe3e8", 240), "Neptuno": (30.1, 1.8, 13, "#4f6df5", 330)}

def punto_orbita(a, inc_deg, ang):
    d = a ** 0.6; inc = np.radians(inc_deg)
    return d*np.cos(ang), d*np.sin(ang)*np.cos(inc), d*np.sin(ang)*np.sin(inc)

def sistema_solar():
    fig = go.Figure(); t = np.linspace(0, 2*np.pi, 300)
    # traza 0 = Sol; luego, por planeta: órbita (1+2i) y planeta (2+2i)
    fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode="markers+text", name="Sol", text=["Sol"],
        textposition="top center", textfont=dict(color="white"), marker=dict(size=26, color="#ffb300")))
    for n, (a, inc, tam, col, ang) in planetas.items():
        ox, oy, oz = punto_orbita(a, inc, t)
        fig.add_trace(go.Scatter3d(x=ox, y=oy, z=oz, mode="lines",
            line=dict(color="rgba(150,150,150,0.35)", width=1), hoverinfo="skip", showlegend=False))
        px, py, pz = punto_orbita(a, inc, np.radians(ang))
        fig.add_trace(go.Scatter3d(x=[px], y=[py], z=[pz], mode="markers+text", name=n, text=[n],
            textposition="top center", textfont=dict(color="white", size=11),
            marker=dict(size=tam, color=col),
            hovertemplate=f"<b>{n}</b><br>Haz clic para entrar al planeta<extra></extra>"))
    fig = estilo_oscuro(fig, "Sistema Solar (distancias no a escala)")
    return fig.update_layout(showlegend=False)

# ---------------- DATOS DE PLANETAS (valores de referencia aproximados, sin verificar) ----------------
def gigante(nubes):
    return [["Termosfera y exosfera", "Capa externa, muy tenue y caliente"],
            ["Estratosfera", "Sobre la tropopausa; neblinas de hidrocarburos"],
            ["Troposfera", "Hasta ~0,1 bar y más abajo; nubes de " + nubes],
            ["Interior profundo", "Sin superficie sólida; el gas se vuelve cada vez más denso"]]

DATOS = {
 "Mercurio": [["4.879 km", "57,9 M km (0,39 UA)", "88 días terrestres", "59 días terrestres", "0", "3,7 m/s²",
              "167 °C", "Casi nula (O, Na, H, He)"],
              [["Exosfera tenue", "Desde la superficie; gas casi inexistente"]]],
 "Venus": [["12.104 km", "108,2 M km (0,72 UA)", "225 días terrestres", "243 días (retrógrado)", "0", "8,9 m/s²",
            "464 °C", "CO₂ 96,5 %, N₂ 3,5 %"],
           [["Termosfera y exosfera", "Por encima de ~120 km"], ["Mesosfera", "~65–120 km"],
            ["Troposfera", "0–~65 km; incluye nubes de ácido sulfúrico (~48–70 km)"]]],
 "Tierra": [["12.742 km", "149,6 M km (1 UA)", "365,25 días", "24 horas", "1", "9,8 m/s²", "15 °C",
             "N₂ 78 %, O₂ 21 %, Ar 0,9 %"],
            [["Exosfera", "Sobre ~700 km, se funde con el espacio"], ["Termosfera", "~85–700 km; auroras"],
             ["Mesosfera", "~50–85 km; aquí se desintegran los meteoros"],
             ["Estratosfera", "~12–50 km; capa de ozono (~15–35 km)"], ["Troposfera", "0–~12 km; clima y nubes"]]],
 "Marte": [["6.779 km", "227,9 M km (1,52 UA)", "687 días terrestres", "24,6 horas", "2", "3,7 m/s²", "-65 °C",
            "CO₂ 95 %, N₂ 2,8 %, Ar 2 %"],
           [["Exosfera", "Sobre ~200 km"], ["Termosfera", "~100–200 km"], ["Mesosfera", "~40–100 km"],
            ["Troposfera", "0–~40 km; polvo y nubes tenues"]]],
 "Júpiter": [["139.820 km", "778 M km (5,2 UA)", "11,9 años", "9,9 horas", "Más de 90", "24,8 m/s²",
              "-110 °C (nivel de 1 bar)", "H₂ ~90 %, He ~10 %"],
             gigante("amoníaco, hidrosulfuro de amonio y agua")],
 "Saturno": [["116.460 km", "1.434 M km (9,54 UA)", "29,4 años", "10,7 horas", "Más de 140", "10,4 m/s²",
              "-140 °C (nivel de 1 bar)", "H₂ ~96 %, He ~3 %"],
             gigante("amoníaco, hidrosulfuro de amonio y agua")],
 "Urano": [["50.724 km", "2.871 M km (19,2 UA)", "84 años", "17,2 horas (retrógrado)", "~28", "8,9 m/s²",
            "-195 °C", "H₂ 83 %, He 15 %, CH₄ 2,3 %"],
           gigante("metano; más abajo, sulfuro de hidrógeno y amoníaco")],
 "Neptuno": [["49.244 km", "4.495 M km (30,1 UA)", "165 años", "16,1 horas", "~16", "11,2 m/s²", "-200 °C",
              "H₂ 80 %, He 19 %, CH₄ 1,5 %"],
             gigante("metano; más abajo, sulfuro de hidrógeno y amoníaco")]}
INFO = {n: {"c": planetas[n][3], "d": v[0], "capas": v[1]} for n, v in DATOS.items()}

# ---------------- PARTÍCULAS (CALCULADO) Y ÓRBITAS ----------------
RADIO_KM = {"Mercurio": 2439.7, "Venus": 6051.8, "Tierra": 6371.0, "Marte": 3389.5,
            "Júpiter": 69911.0, "Saturno": 58232.0, "Urano": 25362.0, "Neptuno": 24622.0}
PERIODO_ANIOS = {"Mercurio": 0.241, "Venus": 0.615, "Tierra": 1.0, "Marte": 1.881,
                 "Júpiter": 11.86, "Saturno": 29.46, "Urano": 84.01, "Neptuno": 164.8}
FLUJO_TIERRA = 6.5e10   # neutrinos solares por cm² por segundo en la Tierra (arXiv hep-ex/0312045)

for n in INFO:
    a = float(planetas[n][0])
    flujo = FLUJO_TIERRA / a**2                                    # baja con el cuadrado de la distancia
    INFO[n]["flujo"] = float(flujo)                                # por cm² por segundo
    INFO[n]["total"] = float(flujo * np.pi * (RADIO_KM[n] * 1e5) ** 2)   # por segundo, a través del planeta
    INFO[n]["atm"] = (n != "Mercurio")
    INFO[n]["orb"] = [a, float(planetas[n][1]), float(planetas[n][4]), PERIODO_ANIOS[n]]

# ---------------- CONTINENTES REALES (Natural Earth, dominio público) ----------------
# Se descarga el mapa de tierras emergidas y se convierte en una imagen blanco/negro
# (blanco = tierra, negro = océano) que se incrusta en el HTML. Si prefieres no depender
# de internet, sube ne_50m_land.geojson o ne_110m_land.geojson a /content y se usa primero.
FUENTES_TIERRA = [
    "/content/ne_50m_land.geojson",
    "/content/ne_110m_land.geojson",
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_50m_land.geojson",
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_land.geojson",
    "https://raw.githubusercontent.com/martynafford/natural-earth-geojson/master/50m/physical/ne_50m_land.json",
    "https://raw.githubusercontent.com/martynafford/natural-earth-geojson/master/110m/physical/ne_110m_land.json",
]

def _leer_fuente(src):
    if src.startswith(("http", "file:")):
        req = urllib.request.Request(src, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=40) as r:
            return json.loads(r.read().decode("utf-8"))
    with open(src, encoding="utf-8") as f:
        return json.load(f)

def mascara_tierra(ancho=1024, alto=512, ss=2):
    for src in FUENTES_TIERRA:
        if not src.startswith(("http", "file:")) and not os.path.exists(src):
            continue
        try:
            gj = _leer_fuente(src)
            img = Image.new("L", (ancho * ss, alto * ss), 0)
            dib = ImageDraw.Draw(img)
            def px(anillo):
                return [((p[0] + 180.0) / 360.0 * ancho * ss, (90.0 - p[1]) / 180.0 * alto * ss) for p in anillo]
            feats = gj["features"] if gj.get("type") == "FeatureCollection" else [gj]
            n_pol = 0
            for ft in feats:
                g = ft.get("geometry", ft)
                tipo, c = g.get("type"), g.get("coordinates")
                if tipo == "Polygon":
                    pols = [c]
                elif tipo == "MultiPolygon":
                    pols = c
                else:
                    continue
                for pol in pols:
                    dib.polygon(px(pol[0]), fill=255)
                    n_pol += 1
                    for hueco in pol[1:]:
                        dib.polygon(px(hueco), fill=0)
            if n_pol == 0:
                raise ValueError("el archivo no tiene polígonos")
            img = img.resize((ancho, alto), Image.LANCZOS)
            buf = io.BytesIO()
            img.save(buf, "PNG", optimize=True)
            print(f"Continentes reales cargados desde {src} ({n_pol} polígonos).")
            return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
        except Exception as e:
            print("No pude usar", src, "->", e)
    print("AVISO: no se pudo obtener el mapa de continentes; la Tierra usará un patrón generado (no real).")
    return None

MASCARA = mascara_tierra()

# ---------------- TEXTURAS REALES DE LOS PLANETAS (Solar System Scope, CC BY 4.0) ----------------
# Texturas basadas en imágenes de la NASA. Licencia CC BY 4.0: hay que dar crédito (la página lo muestra).
# Se descargan al ejecutar la celda y se incrustan en el HTML. Si ya las tienes, sube los archivos
# (2k_mars.jpg, 2k_jupiter.jpg, etc.) a /content y se usan primero. Si algo falla, ese planeta usa
# un aspecto generado por código y se avisa en la salida de la celda.
CARPETA_TEX = "/content"
BASE_TEX = "https://www.solarsystemscope.com/textures/download/"
ANCHO_TEX = 2048          # baja a 1024 si quieres un HTML más liviano
ARCHIVOS_TEX = {
    "Mercurio": ["2k_mercury.jpg"],
    "Venus":    ["2k_venus_atmosphere.jpg", "2k_venus_surface.jpg"],
    "Tierra":   ["2k_earth_daymap.jpg"],
    "Marte":    ["2k_mars.jpg"],
    "Júpiter":  ["2k_jupiter.jpg"],
    "Saturno":  ["2k_saturn.jpg"],
    "Urano":    ["2k_uranus.jpg"],
    "Neptuno":  ["2k_neptune.jpg"],
    "anillos":  ["2k_saturn_ring_alpha.png"],
}

def _abrir_imagen(nombre):
    ruta = os.path.join(CARPETA_TEX, nombre)
    if os.path.exists(ruta):
        return Image.open(ruta)
    if not BASE_TEX:
        raise FileNotFoundError(nombre)
    req = urllib.request.Request(BASE_TEX + nombre, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=40) as r:
        return Image.open(io.BytesIO(r.read()))

def texturas_reales():
    res = {}
    for clave, nombres in ARCHIVOS_TEX.items():
        for nom in nombres:
            try:
                im = _abrir_imagen(nom)
                buf = io.BytesIO()
                if clave == "anillos":
                    im = im.convert("RGBA")
                    if im.width > ANCHO_TEX:
                        im = im.resize((ANCHO_TEX, max(1, round(im.height * ANCHO_TEX / im.width))), Image.LANCZOS)
                    im.save(buf, "PNG", optimize=True); mime = "image/png"
                else:
                    im = im.convert("RGB").resize((ANCHO_TEX, ANCHO_TEX // 2), Image.LANCZOS)
                    im.save(buf, "JPEG", quality=80, optimize=True); mime = "image/jpeg"
                res[clave] = "data:%s;base64,%s" % (mime, base64.b64encode(buf.getvalue()).decode())
                print("Textura real OK:", clave, "(%s)" % nom)
                break
            except Exception as e:
                print("No pude obtener", nom, "->", e)
        else:
            print("AVISO: sin textura real para", clave, "-> se usará un aspecto generado por código.")
    return res

TEXREAL = texturas_reales()

# ---------------- HTML / CSS / JAVASCRIPT ----------------
fig1, fig2 = via_lactea(), sistema_solar()
div1 = fig1.to_html(full_html=False, include_plotlyjs="cdn", div_id="galaxia", default_height="100%")
div2 = fig2.to_html(full_html=False, include_plotlyjs=False, div_id="solar", default_height="100%")

HTML = r"""<!DOCTYPE html><html lang="es"><head><meta charset="utf-8"><title>Atlas de Partículas Cósmicas</title>
<meta name="viewport" content="width=device-width,initial-scale=1">
<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<style>
html,body{margin:0;height:100%;background:#000;color:#fff;font-family:'Segoe UI',system-ui,sans-serif;overflow:hidden}
.pantalla{position:absolute;inset:0}
.fondo{background:radial-gradient(1px 1px at 20% 30%,#fff,transparent),radial-gradient(1px 1px at 70% 20%,#fff,transparent),radial-gradient(2px 2px at 40% 70%,#cfe3ff,transparent),radial-gradient(1px 1px at 85% 80%,#fff,transparent),radial-gradient(1px 1px at 10% 85%,#ffe9b0,transparent),radial-gradient(ellipse at center,#0b0b24 0%,#000 70%);display:flex;flex-direction:column;align-items:center;justify-content:center;text-align:center}
.titulo{font-size:40px;letter-spacing:6px;font-weight:300;margin:0 0 10px}
#carga{transition:opacity .6s}#marco{width:280px;height:6px;background:#222;border-radius:3px;overflow:hidden;margin-top:20px}
#barra{width:0;height:100%;background:linear-gradient(90deg,#2b4cff,#9ecbff)}#pct{margin-top:14px;font-size:14px;color:#aaa}
.sub{color:#9aa;margin:0 0 40px;letter-spacing:3px}
.grande{background:#2b4cff;color:#fff;border:none;padding:16px 44px;font-size:18px;border-radius:30px;cursor:pointer;margin:6px 10px;box-shadow:0 0 25px rgba(43,76,255,.6);transition:transform .15s}
.grande:hover{transform:scale(1.07)}.grande.alt{background:#111;border:1px solid #5a6bd8;box-shadow:none}
.btn{position:fixed;top:12px;left:12px;z-index:11;display:none;background:#111;color:#fff;border:1px solid #444;padding:8px 18px;border-radius:20px;cursor:pointer;font-size:14px}
.btn:hover{background:#2b4cff;border-color:#2b4cff}
#pista{position:fixed;bottom:16px;left:50%;transform:translateX(-50%);z-index:10;font-size:14px;color:#aaa;pointer-events:none;display:none}
.panel{position:fixed;top:60px;width:270px;background:rgba(10,12,30,.9);border:1px solid #2a3355;border-radius:12px;padding:14px;z-index:10;display:none;max-height:calc(100% - 150px);overflow:auto;font-size:13px}
#der{right:12px}#izq{left:12px}.panel h2{margin:0 0 8px;font-weight:400;letter-spacing:2px}.panel h3{margin:0 0 2px;font-weight:400}
.panel table{width:100%;border-collapse:collapse}.panel td{padding:5px 0;border-bottom:1px solid #1d2442;vertical-align:top}.panel td:first-child{color:#8fa0d8;width:42%}
.capa{border-left:10px solid;background:#12162c;border-radius:6px;padding:9px 10px;margin:6px 0;font-size:12px;transition:background .2s}
.capa.act{background:#242c58;outline:1px solid #cfd8ff}
.ice{width:100%;margin-top:12px;background:#2b4cff;color:#fff;border:none;padding:10px;border-radius:20px;cursor:pointer;font-size:14px}
.pag{overflow:auto;background:#03040d}.cont{max-width:900px;margin:0 auto;padding:70px 20px 50px;text-align:left}
.cont h1{font-weight:300;letter-spacing:4px}.cont h2{font-weight:400;color:#9ecbff;margin-top:34px}
.tarj{background:#0d1126;border:1px solid #2a3355;border-radius:12px;padding:14px 18px;margin:10px 0}
.fila{display:flex;gap:12px;flex-wrap:wrap}.fila .tarj{flex:1;min-width:230px}
.oro{color:#ffcc33}.bronce{color:#d9915a}.mapa{width:100%;border-radius:10px}.cont a{color:#9ecbff}
.cont small,.cont li{color:#b9c2e6}.aviso{border-left:4px solid #ffb300;padding-left:12px;color:#ffd98a}
.ins{display:inline-block;font-size:10px;padding:1px 7px;border-radius:10px;margin-top:3px}
.med{background:#1f6f43}.cal{background:#8a6d10}.mod{background:#5b3d99}
#cv{width:100%;height:150px;background:#05071a;border-radius:8px;margin-top:8px}
#ctrl{position:fixed;bottom:14px;left:50%;transform:translateX(-50%);z-index:10;display:none;
 background:rgba(10,12,30,.92);border:1px solid #2a3355;border-radius:20px;padding:8px 16px;
 font-size:13px;align-items:center;gap:10px}
#c3{display:block;width:100%;height:100%;cursor:grab;touch-action:none}
#migas{position:fixed;top:10px;left:50%;transform:translateX(-50%);z-index:10;text-align:center;pointer-events:none;display:none;max-width:44%}
#mg1{font-size:15px;letter-spacing:1px}#mg2{font-size:12px;color:#aeb8e6;margin-top:4px;min-height:16px}
#zbar{position:fixed;bottom:14px;left:50%;transform:translateX(-50%);z-index:10;display:none;
 background:rgba(10,12,30,.92);border:1px solid #2a3355;border-radius:20px;padding:8px 18px;
 font-size:13px;align-items:center;gap:12px;flex-wrap:wrap;justify-content:center;max-width:94%}
#zbar input{width:240px}#zbar small{color:#8e98c8;flex-basis:100%;text-align:center}
#credito{position:fixed;left:14px;bottom:10px;z-index:10;font-size:11px;color:#7f89b8;display:none;max-width:34%;pointer-events:none}
#aviso3d{position:fixed;bottom:60px;left:50%;transform:translateX(-50%);z-index:12;background:#3a2a0a;border:1px solid #ffb300;color:#ffe2a6;padding:8px 14px;border-radius:10px;font-size:13px;display:none}
@media (max-width:760px){.panel{width:44%;font-size:11px;padding:8px}#migas{max-width:60%}}
</style></head><body>

<div id="carga" class="pantalla fondo"><div class="titulo">ATLAS DE PARTÍCULAS CÓSMICAS</div>
<div id="marco"><div id="barra"></div></div><div id="pct">Cargando… 0%</div></div>

<div id="menu" class="pantalla fondo" style="display:none">
<h1 class="titulo" style="font-size:46px">ATLAS DE PARTÍCULAS CÓSMICAS</h1><p class="sub">DETECTOR DE PARTÍCULAS</p>
<div><button class="grande alt" onclick="abrirIce('menu')">IceCube</button>
<button class="grande" onclick="mostrar('v1')">Esquema 3D</button></div></div>

<div id="ice" class="pantalla pag" style="display:none">
<button class="btn" style="display:block" onclick="volverIce()">← Volver</button>
<div class="cont"><h1>ICECUBE</h1>
<p>Detector de neutrinos de un kilómetro cúbico de hielo, en el Polo Sur geográfico. Detecta la luz que dejan las partículas cargadas cuando un neutrino choca con el hielo.</p>
<div class="fila"><div class="tarj"><b>Ubicación</b><br><small>Polo Sur geográfico (Antártida)</small></div>
<div class="tarj"><b>Tamaño</b><br><small>~1 km³ de hielo; unas 86 cuerdas con 5.160 sensores (de memoria, verificar)</small></div></div>

<h2>Alertas públicas en tiempo real</h2>
<div class="fila"><div class="tarj"><b class="oro">● GOLD</b><br><small>Pureza astrofísica ~50 %. Unas 10 alertas por año.</small></div>
<div class="tarj"><b class="bronce">● BRONCE</b><br><small>Pureza astrofísica ~30 %. Unas 17 alertas por año.</small></div></div>
<div class="tarj"><b>Qué trae cada alerta</b><ul><li>Fecha y hora de llegada</li><li>Dirección en el cielo: ascensión recta (RA) y declinación (Dec)</li><li>Error angular de la dirección</li><li>Energía estimada del muón</li><li>Probabilidad de ser astrofísica y tasa de falsas alarmas</li></ul></div>
<p class="aviso">Esta página NO recibe alertas en vivo: se emiten por GCN Kafka y un navegador no puede conectarse directo. Para tiempo real hace falta un servidor intermedio.</p>
<div class="tarj"><small>Tópico Kafka: gcn.notices.icecube.gold_bronze_track_alerts ·
<a href="https://gcn.nasa.gov/missions/icecube" target="_blank">gcn.nasa.gov/missions/icecube</a></small></div>

<h2>Alerta histórica de ejemplo</h2>
<div class="tarj"><b class="oro">IceCube-170922A</b><table style="width:100%;margin-top:6px"><tr><td>Fecha</td><td>22 de septiembre de 2017</td></tr>
<tr><td>Energía</td><td>~290 TeV</td></tr><tr><td>Dirección</td><td>RA 77,43° · Dec +5,72°</td></tr>
<tr><td>Fuente asociada</td><td>Blazar TXS 0506+056</td></tr></table>
<svg viewBox="0 0 360 180" class="mapa" style="margin-top:10px"><rect width="360" height="180" fill="#050818" stroke="#2a3355"/>
<g stroke="#1b2442"><line x1="60" y1="0" x2="60" y2="180"/><line x1="120" y1="0" x2="120" y2="180"/><line x1="180" y1="0" x2="180" y2="180"/>
<line x1="240" y1="0" x2="240" y2="180"/><line x1="300" y1="0" x2="300" y2="180"/><line x1="0" y1="45" x2="360" y2="45"/>
<line x1="0" y1="135" x2="360" y2="135"/></g><line x1="0" y1="90" x2="360" y2="90" stroke="#34406e"/>
<circle cx="77.43" cy="84.28" r="4" fill="#ffb300"/><text x="86" y="82" fill="#fff" font-size="8">170922A</text>
<text x="4" y="10" fill="#6f7bb0" font-size="7">Mapa del cielo (RA 0–360°, Dec −90 a +90°)</text></svg></div>

<h2>Datos públicos</h2>
<ul><li>Catálogo ICECAT-1: 275 eventos de probable origen astrofísico (2011–2020).</li>
<li>Datos por evento IceTracks-DR2 (2008–2022).</li><li>Los datos dan <b>direcciones</b>, no distancias ni el cuerpo emisor.</li>
<li>Las muestras grandes están dominadas por neutrinos y muones atmosféricos.</li></ul>
<p><small>Más información: <a href="https://icecube.wisc.edu/science/data" target="_blank">icecube.wisc.edu/science/data</a>. Revisa su política de uso de datos.</small></p>
</div></div>

<button id="b_ini" class="btn" onclick="mostrar('menu')">← Inicio</button>
<button id="b_gal" class="btn" onclick="mostrar('v1')">← Galaxia</button>
<button id="b_sis" class="btn" onclick="mostrar('v2')">← Sistema Solar</button>
<div id="pista">Arrastra para girar · Rueda para acercar · Haz clic en el Sol ☀</div>
<div id="v1" class="pantalla" style="display:none">%%DIV1%%</div>
<div id="v2" class="pantalla" style="display:none">%%DIV2%%</div>
<div id="v3" class="pantalla" style="display:none"><canvas id="c3"></canvas></div>

<div id="migas"><div id="mg1"></div><div id="mg2"></div></div>
<div id="ctrl">
  <button class="ice" style="width:auto;margin:0;padding:6px 14px" id="pp" onclick="pausa()">⏸ Pausar</button>
  <span>Velocidad</span><input id="vel" type="range" min="0" max="100" value="20" oninput="actVel()">
  <span id="velt"></span><span class="ins mod">MODELADO · tiempo acelerado</span>
</div>
<div id="zbar"><span>Alejar</span><input id="zs" type="range" min="0" max="1000" value="200"><span>Descender por las capas</span>
  <small>Arrastra para girar · Rueda para acercar · Tamaño y espesor de las capas exagerados, no a escala</small></div>
<div id="credito"></div>
<div id="aviso3d"></div>
<div id="izq" class="panel"></div><div id="der" class="panel"></div>

<script>
var INFO=%%DATOS%%;
var MASCARA=%%MASCARA%%,MIMG=null;
if(MASCARA){MIMG=new Image();MIMG.src=MASCARA}
function mascaraLista(){return !!(MIMG&&MIMG.complete&&MIMG.naturalWidth>0)}
var TEXREAL=%%TEXREAL%%,IMG={};
Object.keys(TEXREAL).forEach(function(k){var im=new Image();im.src=TEXREAL[k];IMG[k]=im});
function imgLista(k){var im=IMG[k];return !!(im&&im.complete&&im.naturalWidth>0)}
var ETQ=["Diámetro","Distancia al Sol","Duración del año","Duración del día","Lunas conocidas","Gravedad","Temp. media","Atmósfera"];
var P=['carga','menu','ice','v1','v2','v3'],origen='menu',actual='carga',planetaActual=null,no3D=false;
function $(i){return document.getElementById(i)}

function mostrar(id){
  actual=id;
  P.forEach(function(p){$(p).style.display=(p===id)?((p==='carga'||p==='menu')?'flex':'block'):'none'});
  $('b_ini').style.display=(id==='v1')?'block':'none';
  $('b_gal').style.display=(id==='v2')?'block':'none';
  $('b_sis').style.display=(id==='v3')?'block':'none';
  $('pista').style.display=(id==='v1')?'block':'none';
  $('ctrl').style.display=(id==='v2')?'flex':'none';
  $('zbar').style.display=(id==='v3')?'flex':'none';
  $('migas').style.display=(id==='v3')?'block':'none';
  $('credito').style.display=(id==='v3')?'block':'none';
  animar(id==='v2');
  $('izq').style.display='none';$('der').style.display='none';
  if(id==='v1')Plotly.Plots.resize($('galaxia'));
  if(id==='v2')Plotly.Plots.resize($('solar'));
}
function abrirIce(o){origen=o;mostrar('ice')}
function volverIce(){if(origen==='v2'||origen==='v3')verPlaneta('Tierra');else mostrar('menu')}
function cient(x){var e=Math.floor(Math.log10(x)),m=(x/Math.pow(10,e)).toFixed(1).replace('.',',');return m+' × 10<sup>'+e+'</sup>'}
function avisar(t){var a=$('aviso3d');a.textContent=t;a.style.display='block';setTimeout(function(){a.style.display='none'},6000)}

// ---------- Planetas orbitando (cámara rápida) ----------
var anim=null,tAnios=0,vel=0.04,enPausa=false;
function actVel(){var v=+$('vel').value;vel=Math.pow(10,(v/100)*3-2);
  $('velt').textContent=(vel<1?vel.toFixed(2):vel.toFixed(1))+' años/s'}
function pausa(){enPausa=!enPausa;$('pp').textContent=enPausa?'▶ Reanudar':'⏸ Pausar'}
function pos(o,t){
  var ang=o[2]*Math.PI/180+2*Math.PI*t/o[3],d=Math.pow(o[0],0.6),inc=o[1]*Math.PI/180;
  return [d*Math.cos(ang),d*Math.sin(ang)*Math.cos(inc),d*Math.sin(ang)*Math.sin(inc)];
}
function animar(on){
  if(anim){clearInterval(anim);anim=null}
  if(!on)return;
  actVel();var ult=performance.now();
  anim=setInterval(function(){
    var ahora=performance.now(),dt=(ahora-ult)/1000;ult=ahora;
    if(enPausa)return;
    tAnios+=dt*vel;
    var X=[],Y=[],Z=[],idx=[];
    Object.keys(INFO).forEach(function(n,i){
      var q=pos(INFO[n].orb,tAnios);X.push([q[0]]);Y.push([q[1]]);Z.push([q[2]]);idx.push(2+2*i)});
    Plotly.restyle($('solar'),{x:X,y:Y,z:Z},idx);
  },60);
}

// ---------- Capas de la atmósfera: geometría común (paneles y 3D) ----------
var PAL=['#8d7bff','#4f8dff','#3fd0c9','#78e08f','#f0c05a'],COL_INT='#a05a8c';
function geomCapas(n){
  var caps=INFO[n].capas,m=0,inter=-1;
  caps.forEach(function(c,i){if(/^Interior/.test(c[0]))inter=i;else m++});
  var dz=m>=4?0.13:(m>=2?0.16:0.2),rOut=[],cols=[];
  for(var i=0;i<m;i++){rOut.push(1+(m-i)*dz);cols.push(PAL[m<=1?0:Math.round(i*4/(m-1))])}
  if(inter>=0)cols.push(COL_INT);
  return {m:m,inter:inter,dz:dz,rOut:rOut,cols:cols,rEnd:inter>=0?0.8:1+0.4*dz}
}
function capaEn(g,r){
  if(r>g.rOut[0])return -1;
  for(var i=0;i<g.m;i++){if(r<=g.rOut[i]&&r>g.rOut[i]-g.dz)return i}
  return g.inter>=0?g.inter:-2;
}

// ---------- Animación de partículas del panel (ilustrativa) ----------
var cvAnim=null;
function particulas(n){
  if(cvAnim){cancelAnimationFrame(cvAnim);cvAnim=null}
  var cv=$('cv');if(!cv)return;
  var c=cv.getContext('2d'),W=cv.width=Math.max(300,cv.clientWidth*2),H=cv.height=300,
      cx=W/2,cy=H/2,R=55,A=INFO[n].atm?R+16:R,ps=[],f=[];
  function nueva(){ps.push({r:Math.random()<0.3,y:cy-A*1.3+Math.random()*A*2.6,x:-20,v:2.5+Math.random()*2})}
  for(var i=0;i<6;i++)nueva();
  (function dib(){
    if(!$('cv')||$('der').style.display==='none'){cvAnim=null;return}
    c.clearRect(0,0,W,H);
    if(INFO[n].atm){c.fillStyle='rgba(120,170,255,.18)';c.beginPath();c.arc(cx,cy,A,0,7);c.fill()}
    c.fillStyle=INFO[n].c;c.beginPath();c.arc(cx,cy,R,0,7);c.fill();
    if(Math.random()<0.25)nueva();
    ps=ps.filter(function(p){
      p.x+=p.v;var dy=p.y-cy;
      if(p.r&&Math.abs(dy)<A){var ex=cx-Math.sqrt(A*A-dy*dy);
        if(p.x>=ex){for(var k=0;k<5;k++){var a=-0.9+Math.random()*1.8;
          f.push({x:ex,y:p.y,vx:Math.cos(a)*2.2,vy:Math.sin(a)*2.2,l:20})}return false}}
      c.strokeStyle=p.r?'#ffcc33':'rgba(150,210,255,.8)';c.lineWidth=p.r?3:1.5;
      c.beginPath();c.moveTo(p.x-18,p.y);c.lineTo(p.x,p.y);c.stroke();
      return p.x<W+20});
    f=f.filter(function(q){q.x+=q.vx;q.y+=q.vy;q.l--;
      c.strokeStyle='rgba(255,255,255,'+q.l/20+')';c.lineWidth=1.5;
      c.beginPath();c.moveTo(q.x-q.vx*3,q.y-q.vy*3);c.lineTo(q.x,q.y);c.stroke();return q.l>0});
    cvAnim=requestAnimationFrame(dib);
  })();
}

// ---------- Paneles del planeta (derecha: datos · izquierda: capas) ----------
function panelPlaneta(n){
  var p=INFO[n],g=geomCapas(n),h=`<h2>${n}</h2><table>`;
  for(var i=0;i<ETQ.length;i++)h+=`<tr><td>${ETQ[i]}</td><td>${p.d[i]}</td></tr>`;
  h+=`</table><small style="color:#778">Valores de referencia aproximados</small>`;
  if(n==='Tierra')h+=`<button class="ice" onclick="abrirIce('${actual}')">IceCube</button>`;
  h+=`<h3 style="margin-top:14px">Partículas</h3><table>
    <tr><td>Neutrinos solares por cm² cada segundo<br><span class="ins cal">CALCULADO</span></td><td>${cient(p.flujo)}</td></tr>
    <tr><td>Neutrinos que lo atraviesan por segundo<br><span class="ins cal">CALCULADO</span></td><td>${cient(p.total)}</td></tr></table>
    <canvas id="cv"></canvas><span class="ins mod">MODELADO</span>
    <small style="color:#778"> Base: 6,5 × 10<sup>10</sup> por cm²/s en la Tierra (arXiv hep-ex/0312045), dividido por la distancia² y multiplicado por el área del planeta. Dibujo ilustrativo y no a escala: los neutrinos (azul) lo cruzan; los rayos cósmicos (dorado) se frenan y generan partículas secundarias.</small>`;
  var L='<h3>Capas atmosféricas <span class="ins mod">MODELADO</span></h3><small>De afuera hacia adentro</small>';
  p.capas.forEach(function(c,i){
    L+=`<div class="capa" id="cap${i}" style="border-left-color:${g.cols[i]}"><b>${c[0]}</b><br>${c[1]}</div>`;
  });
  $('der').innerHTML=h;$('izq').innerHTML=L;$('der').style.display='block';$('izq').style.display='block';
  particulas(n);
}
function verPlaneta(n){
  planetaActual=n;
  if(window.THREE&&!no3D){
    try{mostrar('v3');redim3D();cargar3D(n);panelPlaneta(n);
        if(!T3.raf)T3.raf=requestAnimationFrame(bucle);return}
    catch(e){console.error(e);no3D=true;avisar('La vista 3D no pudo iniciar; se muestran los datos sobre el sistema solar.')}
  }else if(!window.THREE&&!no3D){no3D=true;avisar('Three.js no cargó (¿sin internet?); se muestran los datos sobre el sistema solar.')}
  mostrar('v2');panelPlaneta(n);
}

// ---------- Ruido procedural y texturas de planetas ----------
function hash3(x,y,z){var h=Math.imul(x,374761393)^Math.imul(y,668265263)^Math.imul(z,1440662683);
  h=Math.imul(h^(h>>>13),1274126177);h^=h>>>16;return (h>>>0)/4294967296}
function vnoise(x,y,z){
  var xi=Math.floor(x),yi=Math.floor(y),zi=Math.floor(z),xf=x-xi,yf=y-yi,zf=z-zi;
  var u=xf*xf*(3-2*xf),v=yf*yf*(3-2*yf),w=zf*zf*(3-2*zf);
  function L(a,b,t){return a+(b-a)*t}
  return L(L(L(hash3(xi,yi,zi),hash3(xi+1,yi,zi),u),L(hash3(xi,yi+1,zi),hash3(xi+1,yi+1,zi),u),v),
           L(L(hash3(xi,yi,zi+1),hash3(xi+1,yi,zi+1),u),L(hash3(xi,yi+1,zi+1),hash3(xi+1,yi+1,zi+1),u),v),w)}
function fbm(x,y,z,o){var s=0,a=0.5,f=1,t=0;for(var i=0;i<o;i++){s+=a*vnoise(x*f,y*f,z*f);t+=a;f*=2;a*=0.5}return s/t}
function mez(a,b,t){t=Math.max(0,Math.min(1,t));return [a[0]+(b[0]-a[0])*t,a[1]+(b[1]-a[1])*t,a[2]+(b[2]-a[2])*t]}
function suav(a,b,x){var t=Math.max(0,Math.min(1,(x-a)/(b-a)));return t*t*(3-2*t)}
function colorPlaneta(n,x,y,z,lat,m){
  var h,c,w,b;
  if(n==='Tierra'){
    var tierra=(m===undefined)?(fbm(x*2.3+3,y*2.3,z*2.3,5)>0.52?1:0):m/255,aL=Math.abs(lat);
    var h2=fbm(x*3+3,y*3,z*3,5);
    var oceano=mez([8,34,98],[34,98,178],fbm(x*2+7,y*2,z*2,4));
    var suelo=mez([72,122,62],[138,110,80],suav(0.45,0.7,h2));
    var seco=Math.exp(-Math.pow((aL-0.5)/0.28,2))*suav(0.42,0.62,fbm(x*4+1,y*4,z*4,4));
    suelo=mez(suelo,[200,172,112],seco*0.85);
    suelo=mez(suelo,[238,244,248],suav(1.1,1.3,aL));
    c=mez(oceano,suelo,tierra);
    c=mez(c,[225,236,245],suav(1.32,1.48,aL)*(1-tierra));
    var nb=fbm(x*4+9,y*4,z*4+5,4);
    if(nb>0.58)c=mez(c,[255,255,255],(nb-0.58)*2.4);
    return c}
  if(n==='Mercurio'){
    h=fbm(x*5,y*5,z*5,5);var g=85+95*h;
    if(fbm(x*14,y*14,z*14,2)>0.66)g*=0.72;
    return [g,g*0.97,g*0.92]}
  if(n==='Venus'){
    w=fbm(x*2,y*4,z*2,3);h=fbm(x*3+w*2,y*6,z*3+w*2,4);
    return mez([232,192,120],[188,138,78],h*1.2)}
  if(n==='Marte'){
    h=fbm(x*3,y*3,z*3,5);c=mez([190,98,58],[108,56,40],suav(0.45,0.62,h));
    if(Math.abs(lat)>1.38)c=mez(c,[245,240,235],0.8);return c}
  if(n==='Júpiter'){
    w=fbm(x*3,y*3,z*3,3);b=Math.sin(lat*16+w*3.4)*0.5+0.5;
    c=mez([228,200,156],[158,98,60],b);return mez(c,[245,235,215],fbm(x*10,y*3,z*10,3)*0.25)}
  if(n==='Saturno'){
    w=fbm(x*3,y*3,z*3,3);b=Math.sin(lat*12+w*1.6)*0.5+0.5;
    return mez([236,220,170],[200,168,112],b*0.7)}
  if(n==='Urano'){
    b=Math.sin(lat*9)*0.5+0.5;return mez([148,222,228],[120,200,214],b*0.5+fbm(x*3,y*3,z*3,3)*0.2)}
  w=fbm(x*3,y*3,z*3,3);b=Math.sin(lat*10+w*2.2)*0.5+0.5;      // Neptuno
  c=mez([58,92,220],[28,50,155],b);
  if(fbm(x*7+2,y*3,z*7,3)>0.66)c=mez(c,[225,235,255],0.6);return c
}
function texPlaneta(n){
  if(imgLista(n)){var tr=new THREE.Texture(IMG[n]);tr.needsUpdate=true;return tr}
  var big=(n==='Tierra'&&mascaraLista()),W=big?1024:512,H=big?512:256,cv=document.createElement('canvas');cv.width=W;cv.height=H;
  var c=cv.getContext('2d'),im=c.createImageData(W,H),d=im.data,md=null;
  if(big){var mc=document.createElement('canvas');mc.width=W;mc.height=H;var mx=mc.getContext('2d');
    mx.drawImage(MIMG,0,0,W,H);md=mx.getImageData(0,0,W,H).data}
  for(var py=0;py<H;py++){
    var lat=(0.5-(py+0.5)/H)*Math.PI,cl=Math.cos(lat),sl=Math.sin(lat);
    for(var px=0;px<W;px++){
      var lon=(px+0.5)/W*2*Math.PI,x=cl*Math.cos(lon),z=cl*Math.sin(lon),y=sl;
      var k=(py*W+px)*4,col=colorPlaneta(n,x,y,z,lat,md?md[k]:undefined);
      d[k]=col[0];d[k+1]=col[1];d[k+2]=col[2];d[k+3]=255}}
  c.putImageData(im,0,0);return new THREE.CanvasTexture(cv)
}
function texAnillos(){
  var cv=document.createElement('canvas');cv.width=cv.height=256;var c=cv.getContext('2d');
  for(var r=0;r<128;r+=0.5){
    var f=r/128;if(f<0.55||f>0.99)continue;
    var a=0.25+0.55*Math.abs(Math.sin(f*70))*(f<0.8?1:0.7);if(f>0.86&&f<0.89)a=0.04;
    c.strokeStyle='rgba(225,205,160,'+a.toFixed(2)+')';c.lineWidth=1;c.beginPath();c.arc(128,128,r,0,7);c.stroke()}
  return new THREE.CanvasTexture(cv)
}

// ---------- Vista 3D del planeta (Three.js) ----------
var T3=null,GLOW={Venus:[1,0.8,0.45],'Tierra':[0.3,0.6,1],Marte:[1,0.55,0.35],'Júpiter':[0.9,0.75,0.55],Saturno:[0.95,0.85,0.6],Urano:[0.5,0.9,0.95],Neptuno:[0.3,0.45,1]};
function iniciar3D(){
  var cv=$('c3'),R=new THREE.WebGLRenderer({canvas:cv,antialias:true});
  R.setPixelRatio(Math.min(window.devicePixelRatio||1,2));R.setClearColor(0x000000,1);R.localClippingEnabled=true;
  var S=new THREE.Scene(),C=new THREE.PerspectiveCamera(40,1,0.01,400);
  S.add(new THREE.AmbientLight(0xffffff,0.3));
  var Lz=new THREE.DirectionalLight(0xffffff,1.15);Lz.position.set(5,2,3);S.add(Lz);
  var sp=[];for(var i=0;i<1500;i++){var a=Math.random()*6.2832,b=Math.acos(2*Math.random()-1),r=90+Math.random()*60;
    sp.push(r*Math.sin(b)*Math.cos(a),r*Math.cos(b),r*Math.sin(b)*Math.sin(a))}
  var sg=new THREE.BufferGeometry();sg.setAttribute('position',new THREE.Float32BufferAttribute(sp,3));
  S.add(new THREE.Points(sg,new THREE.PointsMaterial({color:0xffffff,size:1.4,sizeAttenuation:false})));
  var NP=260,pPos=new Float32Array(NP*6),pCol=new Float32Array(NP*6),pg=new THREE.BufferGeometry();
  pg.setAttribute('position',new THREE.BufferAttribute(pPos,3));pg.setAttribute('color',new THREE.BufferAttribute(pCol,3));
  var pl=new THREE.LineSegments(pg,new THREE.LineBasicMaterial({vertexColors:true,transparent:true,opacity:0.95,depthTest:false}));
  pl.frustumCulled=false;pl.renderOrder=20;S.add(pl);
  T3={R:R,S:S,C:C,plane:new THREE.Plane(new THREE.Vector3(0,0,-1),1000),NP:NP,pPos:pPos,pCol:pCol,pg:pg,ps:[],
      tex:{},z:0.2,zs:0.2,th:0.7,ph:1.3,vth:0,vph:0,drag:false,raf:0,t:performance.now(),idx:-9,rCur:0,f:0,
      grupo:null,anillos:null,planeta:null,shells:[],circs:[],glow:null,g:null,n:null,atm:false};
  var lx=0,ly=0;
  cv.addEventListener('pointerdown',function(e){T3.drag=true;lx=e.clientX;ly=e.clientY;cv.style.cursor='grabbing';try{cv.setPointerCapture(e.pointerId)}catch(_){}});
  cv.addEventListener('pointermove',function(e){if(!T3.drag)return;
    var k=T3.zs>0.5?0.004:0.006,dx=e.clientX-lx,dy=e.clientY-ly;lx=e.clientX;ly=e.clientY;
    T3.th-=dx*k;T3.ph-=dy*k;T3.vth=-dx*k*0.4;T3.vph=-dy*k*0.4});
  function soltar(){T3.drag=false;cv.style.cursor='grab'}
  cv.addEventListener('pointerup',soltar);cv.addEventListener('pointercancel',soltar);
  cv.addEventListener('wheel',function(e){e.preventDefault();ponerZ(T3.z-e.deltaY*0.0006)},{passive:false});
  $('zs').addEventListener('input',function(){T3.z=(+$('zs').value)/1000});
}
function ponerZ(v){T3.z=Math.max(0,Math.min(1,v));$('zs').value=Math.round(T3.z*1000)}
function redim3D(){
  if(!T3){if(!window.THREE)return;iniciar3D()}
  var w=window.innerWidth,h=window.innerHeight;T3.R.setSize(w,h,false);T3.C.aspect=w/h;T3.C.updateProjectionMatrix()
}
function liberar(o){o.traverse(function(x){if(x.geometry)x.geometry.dispose();
  if(x.material){if(x.material.map&&x.material.map!==null&&false)x.material.map.dispose();x.material.dispose()}})}
function circulo(r,col,G){
  var pts=[];for(var i=0;i<128;i++){var a=i/128*Math.PI*2;pts.push(new THREE.Vector3(Math.cos(a)*r,Math.sin(a)*r,0))}
  var l=new THREE.LineLoop(new THREE.BufferGeometry().setFromPoints(pts),new THREE.LineBasicMaterial({color:col}));
  l.visible=false;G.add(l);return l
}
function cargar3D(n){
  if(!T3)iniciar3D();
  var p=INFO[n],g=geomCapas(n),pl=T3.plane;
  if(T3.grupo){T3.S.remove(T3.grupo);liberar(T3.grupo)}
  if(T3.anillos){T3.S.remove(T3.anillos);liberar(T3.anillos);T3.anillos=null}
  var G=new THREE.Group();
  var real=imgLista(n),clave=n+(real?'R':'')+((n==='Tierra'&&mascaraLista())?'M':''),tex=T3.tex[clave]||(T3.tex[clave]=texPlaneta(n));
  var mp=new THREE.Mesh(new THREE.SphereGeometry(1,96,64),new THREE.MeshPhongMaterial({map:tex,shininess:6,clippingPlanes:[pl]}));
  G.add(mp);
  G.add(new THREE.Mesh(new THREE.SphereGeometry(0.998,48,32),
    new THREE.MeshBasicMaterial({color:g.inter>=0?'#6b3f66':'#5a3b2c',side:THREE.BackSide,clippingPlanes:[pl]})));
  var shells=[],circs=[];
  for(var i=0;i<g.m;i++){
    var sh=new THREE.Mesh(new THREE.SphereGeometry(g.rOut[i],64,40),
      new THREE.MeshBasicMaterial({color:g.cols[i],transparent:true,opacity:0.17,side:THREE.DoubleSide,depthWrite:false,clippingPlanes:[pl]}));
    sh.visible=false;G.add(sh);shells.push(sh);circs.push(circulo(g.rOut[i],g.cols[i],G))}
  circs.push(circulo(1,'#ffffff',G));
  var glow=null;
  if(p.atm&&GLOW[n]){
    var gc=GLOW[n];
    glow=new THREE.Mesh(new THREE.SphereGeometry(1.13,64,40),new THREE.ShaderMaterial({
      uniforms:{c:{value:new THREE.Color(gc[0],gc[1],gc[2])},k:{value:(n==='Marte'?0.45:0.8)}},
      vertexShader:'varying vec3 vN;void main(){vN=normalize(normalMatrix*normal);gl_Position=projectionMatrix*modelViewMatrix*vec4(position,1.0);}',
      fragmentShader:'varying vec3 vN;uniform vec3 c;uniform float k;void main(){float i=pow(max(0.0,0.72-dot(vN,vec3(0.0,0.0,1.0))),3.0);gl_FragColor=vec4(c*i*k,1.0);}',
      side:THREE.BackSide,blending:THREE.AdditiveBlending,transparent:true,depthWrite:false}));
    G.add(glow)}
  T3.S.add(G);
  var anillosReales=false;
  if(n==='Saturno'){
    var ri=1.3,ro=2.4,tA;
    if(imgLista('anillos')){anillosReales=true;ri=1.24;ro=2.27;tA=new THREE.Texture(IMG['anillos']);tA.needsUpdate=true}else tA=texAnillos();
    var rgG=new THREE.RingGeometry(ri,ro,128);
    if(anillosReales){var pa=rgG.attributes.position,uv=rgG.attributes.uv,vv=new THREE.Vector3();
      for(var q=0;q<pa.count;q++){vv.fromBufferAttribute(pa,q);uv.setXY(q,(vv.length()-ri)/(ro-ri),0.5)}}
    var rg=new THREE.Mesh(rgG,new THREE.MeshBasicMaterial({map:tA,transparent:true,side:THREE.DoubleSide,depthWrite:false}));
    rg.rotation.x=Math.PI/2;var an=new THREE.Group();an.add(rg);an.rotation.z=0.46;T3.S.add(an);T3.anillos=an}
  var cr=[];
  if(real||anillosReales)cr.push('Texturas: Solar System Scope (CC BY 4.0), basadas en imágenes de la NASA');
  if(n==='Tierra'&&!real&&mascaraLista())cr.push('Continentes: Natural Earth (dominio público); colores generados');
  if(!real&&!(n==='Tierra'&&mascaraLista()))cr.push('Aspecto generado por código, no es una imagen real');
  $('credito').textContent=cr.join(' · ');
  T3.grupo=G;T3.planeta=mp;T3.shells=shells;T3.circs=circs;T3.glow=glow;T3.g=g;T3.n=n;T3.atm=!!glow;
  T3.ps.length=0;T3.idx=-9;ponerZ(0.2);T3.zs=0.2;T3.th=0.7;T3.ph=1.3;T3.vth=T3.vph=0;T3.t=performance.now();
  $('mg1').textContent='Sistema Solar › '+n;$('mg2').textContent='Acércate para descender por las capas de la atmósfera'
}
function camara(){
  var z=T3.zs,C=T3.C,g=T3.g,th=T3.th,ph=T3.ph;
  var cd=new THREE.Vector3(Math.sin(ph)*Math.cos(th),Math.cos(ph),Math.sin(ph)*Math.sin(th));
  var right=new THREE.Vector3().crossVectors(new THREE.Vector3(0,1,0),cd).normalize();
  var up=new THREE.Vector3().crossVectors(cd,right).normalize(),tgt=new THREE.Vector3(),dist;
  if(z<=0.5){dist=9-5.8*(z/0.5);T3.f=0;T3.rCur=9}
  else{
    var s=(z-0.5)/0.5;dist=3.2-2.7*s;
    var f=Math.min(1,s/0.1);f=f*f*(3-2*f);
    var r0=g.rOut[0]-0.15*g.dz,rT=r0+(g.rEnd-r0)*s;
    tgt.copy(up).multiplyScalar(rT*f);T3.f=f;T3.rCur=rT*f}
  C.position.copy(tgt).addScaledVector(cd,dist);C.lookAt(tgt);
  T3.plane.normal.copy(cd).negate();T3.plane.constant=(z>0.5)?0:1000;
}
function actualizarCapa(){
  var g=T3.g,cut=T3.zs>0.5,idx=-9;
  if(cut&&T3.f>0.99)idx=capaEn(g,T3.rCur);
  if(idx===T3.idx)return;
  T3.idx=idx;var p=INFO[T3.n];
  p.capas.forEach(function(c,i){var e=$('cap'+i);if(e)e.className='capa'+(i===idx?' act':'')});
  var t='Acércate para descender por las capas de la atmósfera';
  if(cut&&idx===-9)t='Corte transversal: bajando hacia las capas…';
  else if(idx===-1)t='Espacio: por encima de la capa más externa';
  else if(idx===-2)t='Superficie';
  else if(idx>=0)t=p.capas[idx][0]+' · '+p.capas[idx][1];
  $('mg2').textContent=t;
  $('mg1').textContent='Sistema Solar › '+T3.n+((idx>=0)?' › '+p.capas[idx][0]:'')
}
function lanzar(cr,rMax,rAtm){
  var T=T3;if(T.ps.length>=T.NP-40)return;
  var a=Math.random()*6.2832,rr=Math.sqrt(Math.random())*rMax;
  T.ps.push({t:cr?1:0,x:7,y:rr*Math.cos(a),z:rr*Math.sin(a),vx:cr?-4.2:-5.6,vy:0,vz:0,l:0,
    stop:1.02+Math.random()*(Math.max(rAtm,1.1)-1.02)})
}
function pasoParticulas(dt){
  var T=T3,rAtm=T.g.rOut[0]||1.2,rMax=rAtm*1.15,i,q,n;
  n=Math.floor(38*dt+Math.random());for(i=0;i<n;i++)lanzar(false,rMax,rAtm);
  n=Math.floor(7*dt+Math.random());for(i=0;i<n;i++)lanzar(true,rMax,rAtm);
  var nuevos=[];
  T.ps=T.ps.filter(function(q){
    q.x+=q.vx*dt;q.y+=q.vy*dt;q.z+=q.vz*dt;
    if(q.t===2){q.l-=dt;return q.l>0}
    if(q.t===1&&q.x>0&&Math.sqrt(q.x*q.x+q.y*q.y+q.z*q.z)<=q.stop){
      for(var k=0;k<4;k++)nuevos.push({t:2,x:q.x,y:q.y,z:q.z,vx:-1.6-Math.random()*1.4,
        vy:(Math.random()-0.5)*2.2,vz:(Math.random()-0.5)*2.2,l:0.6,stop:0});
      return false}
    return q.x>-8});
  nuevos.forEach(function(q){if(T.ps.length<T.NP)T.ps.push(q)});
  var P=T.pPos,Cc=T.pCol,c,f;
  for(i=0;i<T.NP;i++){
    var o=i*6;
    if(i<T.ps.length){q=T.ps[i];f=(q.t===2)?Math.max(0,q.l/0.6):1;
      P[o]=q.x;P[o+1]=q.y;P[o+2]=q.z;P[o+3]=q.x-q.vx*0.05;P[o+4]=q.y-q.vy*0.05;P[o+5]=q.z-q.vz*0.05;
      c=(q.t===0)?[0.55,0.8,1]:(q.t===1)?[1,0.8,0.2]:[1,1,1];
      Cc[o]=Cc[o+3]=c[0]*f;Cc[o+1]=Cc[o+4]=c[1]*f;Cc[o+2]=Cc[o+5]=c[2]*f}
    else{for(var k=0;k<6;k++){P[o+k]=0;Cc[o+k]=0}}}
  T.pg.attributes.position.needsUpdate=true;T.pg.attributes.color.needsUpdate=true
}
function frame(){
  var T=T3,now=performance.now(),dt=Math.min(0.05,(now-T.t)/1000);T.t=now;
  T.zs+=(T.z-T.zs)*Math.min(1,dt*7);
  if(!T.drag){T.th+=T.vth;T.ph+=T.vph;T.vth*=0.94;T.vph*=0.94;
    if(T.zs<0.5&&Math.abs(T.vth)<0.0005)T.th+=0.12*dt}
  T.ph=Math.max(0.12,Math.min(Math.PI-0.12,T.ph));
  camara();
  var cut=T.zs>0.5;
  if(T.glow)T.glow.visible=!cut;
  T.shells.forEach(function(s){s.visible=cut});
  T.circs.forEach(function(c){c.visible=cut;c.quaternion.copy(T.C.quaternion)});
  if(T.anillos)T.anillos.visible=!cut;
  if(!cut)T.planeta.rotation.y+=0.15*dt;
  actualizarCapa();pasoParticulas(dt);
  T.R.render(T.S,T.C)
}
function bucle(){
  if(actual!=='v3'||!T3){if(T3)T3.raf=0;return}
  T3.raf=requestAnimationFrame(bucle);
  try{frame()}catch(e){console.error(e);cancelAnimationFrame(T3.raf);T3.raf=0;no3D=true;
    avisar('La vista 3D falló; se muestran los datos sobre el sistema solar.');mostrar('v2');panelPlaneta(planetaActual)}
}
document.addEventListener('keydown',function(e){
  if(actual!=='v3'||!T3)return;
  if(e.key==='+'||e.key==='=')ponerZ(T3.z+0.06);
  else if(e.key==='-'||e.key==='_')ponerZ(T3.z-0.06);
  else if(e.key==='Escape')mostrar('v2')
});

// ---------- Carga y eventos de Plotly ----------
var prog=0,listo=false,t0=Date.now();
window.addEventListener('load',function(){
  listo=true;
  $('galaxia').on('plotly_click',function(ev){var p=ev.points[0];if(p&&p.data.name==='Sol')mostrar('v2')});
  $('solar').on('plotly_click',function(ev){var p=ev.points[0];if(p&&INFO[p.data.name])verPlaneta(p.data.name)});
});
var timer=setInterval(function(){
  var lim=listo?100:90;if(prog<lim)prog=Math.min(lim,prog+(listo?4:1.2));
  $('barra').style.width=prog+'%';$('pct').textContent='Cargando… '+Math.floor(prog)+'%';
  if(prog>=100&&Date.now()-t0>2500){clearInterval(timer);var c=$('carga');c.style.opacity=0;
    setTimeout(function(){c.style.opacity=1;mostrar('menu')},600)}
},40);
window.addEventListener('resize',function(){
  if(window.Plotly){Plotly.Plots.resize($('galaxia'));Plotly.Plots.resize($('solar'))}
  if(T3&&actual==='v3')redim3D()});
</script></body></html>"""

HTML = (HTML.replace("%%DIV1%%", div1).replace("%%DIV2%%", div2)
            .replace("%%DATOS%%", json.dumps(INFO, ensure_ascii=False))
            .replace("%%MASCARA%%", json.dumps(MASCARA))
            .replace("%%TEXREAL%%", json.dumps(TEXREAL)))

with open("atlas_particulas.html", "w", encoding="utf-8") as f:
    f.write(HTML)
files.download("atlas_particulas.html")

Continentes reales cargados desde https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_50m_land.geojson (1421 polígonos).
Textura real OK: Mercurio (2k_mercury.jpg)
Textura real OK: Venus (2k_venus_atmosphere.jpg)
Textura real OK: Tierra (2k_earth_daymap.jpg)
Textura real OK: Marte (2k_mars.jpg)
Textura real OK: Júpiter (2k_jupiter.jpg)
Textura real OK: Saturno (2k_saturn.jpg)
Textura real OK: Urano (2k_uranus.jpg)
Textura real OK: Neptuno (2k_neptune.jpg)
Textura real OK: anillos (2k_saturn_ring_alpha.png)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>